In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.distributed as dist
import numpy as np
import gymnasium as gym


In [ ]:
for i in gym.envs.registry:
    print(i)

In [ ]:
env = gym.make("CartPole-v1", render_mode="rgb_array")

In [ ]:
env.observation_space, env.action_space

In [ ]:
env.observation_space.shape[0]

In [ ]:
env.observation_space.low, env.observation_space.high


In [ ]:
for i in range(5):
    print(env.observation_space.sample(), env.action_space.sample())

In [ ]:
obs, info = env.reset()
obs, info

In [ ]:
input_dim = env.observation_space.shape[0]
hidden_dim = 64
output_dim = env.action_space.n

def create_policy_network(input_dim, hidden_dim, output_dim):
    model = nn.Sequential(
        nn.Linear(input_dim, hidden_dim).double(),
        nn.ReLU(),
        nn.Linear(hidden_dim, hidden_dim).double(),
        nn.ReLU(),
        nn.Linear(hidden_dim, output_dim).double(),
        nn.Softmax(dim=-1)
    )
    return model

print(input_dim, hidden_dim, output_dim)

In [ ]:
policy = create_policy_network(input_dim, hidden_dim, output_dim)

In [ ]:
observation, _ = env.reset()
policy(torch.tensor(observation, dtype=torch.float64))

In [ ]:
action = env.action_space.sample(probability=policy(torch.tensor(observation, dtype=torch.float64)).detach().cpu().numpy())
print(action)

In [ ]:
env.step(action)

In [ ]:
policy.train()

# write boiletplate code for training a gymnasium environment using loss function and a discounted reward function
learning_rate = 0.01
gamma = 0.99

optimizer = optim.Adam(policy.parameters(), lr=learning_rate)

# training loop
num_episodes = 500
reward_goal = 200

episode_rewards = []
epoch_frames = []

for episode in range(num_episodes):
    render = episode % 10 == 0
    observation, _ = env.reset()
    done = False
    rewards = []
    log_probs = []
    frames = []
    
    if render:
        frames.append(env.render())

    while not done:
        action_probs = policy(torch.tensor(observation, dtype=torch.float64))
        action = env.action_space.sample(probability=action_probs.detach().cpu().numpy())
        
        log_prob = torch.log(action_probs[action])
        log_probs.append(log_prob)
        
        observation, reward, terminated, truncated, info = env.step(action)
        rewards.append(reward)

        if render:
            epoch_frames.append(env.render())
        
        done = terminated or truncated
    
    # compute discounted rewards
    discounted_rewards = []
    cumulative_reward = 0
    for r in reversed(rewards):
        cumulative_reward = r + gamma * cumulative_reward
        discounted_rewards.insert(0, cumulative_reward)
    
    discounted_rewards = torch.tensor(discounted_rewards, dtype=torch.float64)
    # normalize discounted rewards
    discounted_rewards = (discounted_rewards - discounted_rewards.mean()) / (discounted_rewards.std() + 1e-9)
    log_probs = torch.stack(log_probs)
    
    # compute loss
    loss = -torch.sum(log_probs * discounted_rewards)
    
    # backpropagation
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    episode_rewards.append(sum(rewards))

    mean_reward = np.mean(episode_rewards[-100:])
    if mean_reward >= reward_goal:
        print(f"Solved in episode {episode + 1} with mean reward {mean_reward:.2f} over the last 100 episodes!")
        break

    if episode % 10 == 0:
        print(f"Episode {episode + 1}/{num_episodes}, Loss: {loss.item()}")
        # print total reward for the episode
        total_reward = sum(rewards)
        print(f"Episode {episode + 1}/{num_episodes}, Total Reward: {total_reward}")


policy.eval()

In [ ]:
# plot the episode rewards using a line chart with plotly express
import plotly.express as px
fig = px.line(x=list(range(len(episode_rewards))), y=episode_rewards, title="Episode Rewards", labels={"x": "Episode", "y": "Total Reward"})
fig.show()

In [ ]:
# create a matplotlib animation of the rendered frames
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import display, HTML

def display_frames(frames):
    fig, ax = plt.subplots()
    frame = frames[0]
    im = ax.imshow(frame)

    def update(frame):
        im.set_array(frame)
        return [im]

    ani = animation.FuncAnimation(fig, update, frames=frames, repeat=False)
    plt.close(fig)
    display(HTML(ani.to_jshtml()))


In [ ]:

# render the animation in the notebook
display_frames(epoch_frames)